# SSB Config Admin

Komplett notebook for å administrere `ssb_config`-tabellen.

## 📋 Oversikt

**Bruk:** Sett parametere i neste celle og kjør hele notebooken (Run All).
Steg som har `None`-parametere hoppes automatisk over.

### Steg:
1. **Se alle tabeller** – Oversikt over config
2. **Legg til ny tabell** – Enkeltinnlegg (henter metadata fra SSB API)
3. **Oppdater tabell** – Endre kategori / lookback_periods / priority
4. **Slett tabell** – Fjern fra config
5. **Vis loading-historikk** – Når ble tabeller sist lastet?
6. **Mark as loaded** – Utility-funksjon (bruk i ingest-script)
7. **Reset timestamp** – Force re-load neste gang Update Detector kjører
8. **Reset til default** – Batch-reset av kategori/lookback til API-verdier
9. **Batch insert** – Legg til flere tabeller på én gang
10. **Batch slett** – Fjern flere tabeller fra config på én gang
11. **Reset alle** – Alle tabeller til default (krever confirm=True)

### ssb_config-schema:
| Kolonne | Type | Beskrivelse |
|---|---|---|
| `table_id` | STRING | SSB tabellnummer, PK |
| `table_name` | STRING | Tabellnavn fra SSB API |
| `frequency` | STRING | Annual / Quarterly / Monthly / Weekly / Daily |
| `category` | STRING | Tematisk kategori fra SSB |
| `lookback_periods` | INT | Antall perioder å hente ved oppdatering |
| `priority` | STRING | CRITICAL eller NORMAL (default: NORMAL) |
| `last_downloaded_timestamp` | STRING | ISO-timestamp for siste vellykkede nedlasting (satt av 04) |
| `last_loaded_timestamp` | STRING | ISO-timestamp for siste vellykkede standardisering (satt av 05) |

---


In [1]:
# ============================================================================
# IMPORTS
# ============================================================================
# Standard oppsett – kobler til Spark og peker på ssb_config-tabellen som
# resten av notebooken jobber mot.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta.tables import DeltaTable
from datetime import datetime
from typing import List

spark = SparkSession.builder.appName("SSBConfigAdmin").getOrCreate()

CONFIG_TABLE = "statbank_staging.pipeline.ssb_config"

print("✅ Imports lastet")
print(f"   Konfig-tabell: {CONFIG_TABLE}")


StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 3, Finished, Available, Finished, False)

✅ Imports lastet
   Konfig-tabell: pipeline.ssb_config


In [ ]:
# ============================================================================
# LAST INN METADATA-BIBLIOTEK (01_ssb_metadata_oppsett i LIBRARY_MODE)
# ============================================================================
# Gjør fetch_table_metadata(), parse_cfg_row() og upsert_metadata_row()
# tilgjengelig, slik at ssb_metadata (datakatalogen) bygges opp automatisk
# for hver tabell som legges til/oppdateres i ssb_config.

%run 01_ssb_metadata_oppsett {"LIBRARY_MODE": true}


In [2]:
# ============================================================================
# PARAMETRE – Endre disse verdiene, kjør deretter Run All
# Steg der parameteren er None hoppes automatisk over.
# ============================================================================
# Dette er "fjernkontrollen" for hele notebooken: alle de ti stegene lenger
# ned leser fra parametrene her. Vil du f.eks. bare oppdatere én tabell,
# setter du kun tabell_id_update – de andre stegene ser at deres egne
# parametre er None og hopper automatisk over seg selv.

# --- LEGG TIL NY TABELL (enkelt) ---
tabell_id        = "14289"   # eks: "14306"
kategori         = None    # Override kategori fra API  (None = bruk API-verdi)
lookback         = None    # Override lookback_periods  (None = bruk default)
priority_ny      = None    # "CRITICAL" eller "NORMAL"  (None = "NORMAL")

# --- LEGG TIL FLERE TABELLER (batch) ---
table_list       = []     # eks: ["14305", "14307"]

# --- OPPDATER EKSISTERENDE TABELL ---
tabell_id_update = None    # eks: "01222"
ny_kategori      = None    # Ny kategori       (None = hent default fra API)
ny_lookback      = None    # Ny lookback       (None = hent default fra API)
ny_priority      = None    # "CRITICAL" / "NORMAL" (None = ingen endring)

# --- SLETT TABELL ---
tabell_id_delete = None    # eks: "09429"

# --- SLETT FLERE TABELLER (batch) ---
table_list_delete = []     # eks: ["14305", "14307"]

# --- RESET TIMESTAMP (force re-load) ---
tabell_id_reset  = None    # eks: "07459"

# --- RESET TIL DEFAULT (batch fra liste) ---
table_list_default = []    # eks: ["14305", "14307"]

print("✅ Parametre definert")
print(f"   tabell_id:          {tabell_id}")
print(f"   table_list:         {table_list}")
print(f"   tabell_id_update:   {tabell_id_update}")
print(f"   tabell_id_delete:   {tabell_id_delete}")
print(f"   table_list_delete:  {table_list_delete}")
print(f"   tabell_id_reset:    {tabell_id_reset}")
print(f"   table_list_default: {table_list_default}")

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 4, Finished, Available, Finished, False)

✅ Parametre definert
   tabell_id:          14289
   table_list:         []
   tabell_id_update:   None
   tabell_id_delete:   None
   tabell_id_reset:    None
   table_list_default: []


In [3]:
# ============================================================================
# FUNKSJONER
# ============================================================================
# Selve verktøykassen: her defineres alt notebooken kan gjøre med
# ssb_config (legge til, oppdatere, resette, slette), samt et lite
# "reparasjons"-steg (_ensure_config_columns) som legger til kolonner i
# tabellen hvis en eldre versjon av den mangler dem. De nummererte stegene
# lenger ned i notebooken kaller bare disse funksjonene med parametrene du
# satte over.

VALID_PRIORITIES = {"CRITICAL", "NORMAL"}

SSB_CONFIG_SCHEMA = StructType([
    StructField("table_id",                  StringType(),  False),
    StructField("table_name",                StringType(),  True),
    StructField("frequency",                 StringType(),  True),
    StructField("category",                  StringType(),  True),
    StructField("lookback_periods",          IntegerType(), True),
    StructField("priority",                  StringType(),  True),
    StructField("last_downloaded_timestamp", StringType(),  True),
    StructField("last_loaded_timestamp",     StringType(),  True),
])


def _ensure_config_columns():
    """
    Sikrer at ssb_config har priority- og last_downloaded_timestamp-kolonnene.
    Trygg å kalle flere ganger – gjør ingenting hvis kolonnene allerede finnes.
    """
    try:
        cols = [f.name for f in spark.table(CONFIG_TABLE).schema.fields]
        if "priority" not in cols:
            print("⚙️  Legger til manglende 'priority'-kolonne i ssb_config...")
            spark.sql(f"""
                ALTER TABLE {CONFIG_TABLE}
                ADD COLUMN priority STRING
            """)
            spark.sql(f"""
                UPDATE {CONFIG_TABLE}
                SET priority = 'NORMAL'
                WHERE priority IS NULL
            """)
            print("✅ 'priority'-kolonne lagt til og satt til NORMAL for alle rader")
        if "last_downloaded_timestamp" not in cols:
            print("⚙️  Legger til manglende 'last_downloaded_timestamp'-kolonne i ssb_config...")
            spark.sql(f"""
                ALTER TABLE {CONFIG_TABLE}
                ADD COLUMN last_downloaded_timestamp STRING
            """)
            print("✅ 'last_downloaded_timestamp'-kolonne lagt til")
    except Exception:
        pass  # tabellen finnes kanskje ikke ennå – ok


def get_default_lookback(frequency: str) -> int:
    """
    Standard lookback-perioder basert på frekvens.
    frequency kommer fra 01_ssb_metadata_oppsett (update_frequency /
    update_frequency_inferred) og er derfor på norsk.
    """
    if not frequency:
        return 2
    return {
        "årlig":       2,
        "kvartalsvis": 4,
        "månedlig":    12,
        "ukentlig":    52,
        "daglig":      365,
    }.get(frequency.strip().lower(), 2)


def _table_exists_in_config(table_id: str) -> bool:
    """Sjekk om table_id allerede er i config (parameterisert query)"""
    count = (
        spark.table(CONFIG_TABLE)
        .filter(F.col("table_id") == table_id)
        .count()
    )
    return count > 0


def add_new_table(
    table_id: str,
    override_category: str = None,
    override_lookback: int = None,
    override_priority: str = None,
) -> bool:
    """
    Legg til ny tabell i config (henter metadata fra SSB API via
    01_ssb_metadata_oppsett) og bygger samtidig katalograden i ssb_metadata.

    Returns:
        True hvis lagt til, False hvis allerede finnes eller feil.
    """
    _ensure_config_columns()

    if _table_exists_in_config(table_id):
        print(f"⚠️  Tabell {table_id} finnes allerede i config – hopper over")
        return False

    print(f"📡 Henter metadata for tabell {table_id} fra SSB API...")
    try:
        meta = fetch_table_metadata(table_id)
        meta_row = parse_cfg_row(table_id, meta)
    except Exception as e:
        print(f"❌ Kunne ikke hente metadata for {table_id}: {e}")
        return False

    frequency = meta_row["update_frequency"] or meta_row["update_frequency_inferred"]

    info = {
        "table_id":                  table_id,
        "table_name":                meta_row["title"] or "Ukjent",
        "frequency":                 frequency or "Unknown",
        "category":                  meta_row["subject_area"],
        "lookback_periods":          None,
        "priority":                  "NORMAL",
        "last_downloaded_timestamp": None,
        "last_loaded_timestamp":     None,
    }

    info["lookback_periods"] = (
        override_lookback
        if override_lookback is not None
        else get_default_lookback(frequency)
    )
    if override_category is not None:
        info["category"] = override_category

    raw_priority = (override_priority or "NORMAL").upper()
    info["priority"] = raw_priority if raw_priority in VALID_PRIORITIES else "NORMAL"

    print(f"\n✅ Klar til lagring:")
    for k, v in info.items():
        print(f"   {k}: {v}")

    new_df = spark.createDataFrame([info], schema=SSB_CONFIG_SCHEMA)
    new_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(CONFIG_TABLE)
    print(f"\n✅ Tabell {table_id} lagt til i config!")

    upsert_metadata_row(meta_row)
    print(f"✅ Tabell {table_id} lagt til i ssb_metadata (katalog)!")
    return True


def delete_table(table_id: str) -> bool:
    """
    Slett én tabell fra ssb_config. Rører ikke data som allerede er hentet
    ned eller standardisert – kun styringsraden i ssb_config.

    Returns:
        True hvis slettet, False hvis tabellen ikke fantes.
    """
    if not _table_exists_in_config(table_id):
        print(f"❌ Tabell {table_id} finnes IKKE i config!")
        return False

    DeltaTable.forName(spark, CONFIG_TABLE).delete(
        condition=F.col("table_id") == table_id
    )
    print(f"✅ Tabell {table_id} slettet fra config!")
    return True


def delete_tables(table_ids: List[str]) -> None:
    """
    Slett flere tabeller fra ssb_config i ett kall – samme mønster som
    add_new_table/batch insert, men for sletting.
    """
    if not table_ids:
        print("⚠️  Ingen tabellnumre oppgitt!")
        return

    print(f"📋 Sletter {len(table_ids)} tabeller: {', '.join(table_ids)}\n")
    ok = skipped = 0
    for tid in table_ids:
        print(f"--- {tid} ---")
        if delete_table(tid):
            ok += 1
        else:
            skipped += 1
        print()

    print(f"📊 Resultat: {ok} slettet, {skipped} hoppet over")


def reset_table_to_defaults(table_id: str) -> bool:
    """
    Reset kategori og lookback_periods til ferske verdier fra SSB API,
    og oppdaterer samtidig katalograden i ssb_metadata.
    last_loaded_timestamp og priority røres ikke.
    """
    _ensure_config_columns()

    if not _table_exists_in_config(table_id):
        print(f"❌ Tabell {table_id} finnes IKKE i config!")
        return False

    print(f"📡 Henter frisk metadata fra SSB API for {table_id}...")
    try:
        meta = fetch_table_metadata(table_id)
        meta_row = parse_cfg_row(table_id, meta)
    except Exception as e:
        print(f"❌ Kunne ikke hente metadata for {table_id}: {e}")
        return False

    frequency         = meta_row["update_frequency"] or meta_row["update_frequency_inferred"]
    default_lookback  = get_default_lookback(frequency)
    new_category      = meta_row["subject_area"] or ""

    (
        DeltaTable.forName(spark, CONFIG_TABLE)
        .update(
            condition=F.col("table_id") == table_id,
            set={
                "category":         F.lit(new_category),
                "lookback_periods": F.lit(default_lookback),
            },
        )
    )

    print(f"✅ Tabell {table_id} resettet til default-verdier")
    print(f"   category         → {new_category}")
    print(f"   lookback_periods → {default_lookback}")
    display(spark.table(CONFIG_TABLE).filter(F.col("table_id") == table_id))

    upsert_metadata_row(meta_row)
    print(f"✅ ssb_metadata oppdatert for {table_id}")
    return True


def mark_table_as_loaded(table_id: str):
    """
    Marker tabell som lastet – oppdaterer last_loaded_timestamp til nå.
    Kall denne fra ingest-script etter vellykket last:
        mark_table_as_loaded(table_id)
    """
    timestamp = datetime.now().isoformat()
    (
        DeltaTable.forName(spark, CONFIG_TABLE)
        .update(
            condition=F.col("table_id") == table_id,
            set={"last_loaded_timestamp": F.lit(timestamp)},
        )
    )
    print(f"✅ Tabell {table_id} merket som lastet ({timestamp})")


def reset_all_tables_to_defaults(confirm: bool = False):
    """
    Reset ALLE tabeller i config til default-verdier fra SSB API.
    Args:
        confirm: Må settes til True for å kjøre.
    """
    if not confirm:
        print("⚠️  ADVARSEL: Denne funksjonen resetter ALLE tabeller!")
        print("   For å kjøre: reset_all_tables_to_defaults(confirm=True)")
        return

    table_ids = [
        row["table_id"]
        for row in spark.table(CONFIG_TABLE).select("table_id").orderBy("table_id").collect()
    ]

    if not table_ids:
        print("⚠️  Ingen tabeller funnet i config!")
        return

    print(f"📋 Resetter {len(table_ids)} tabeller: {', '.join(table_ids)}\n")
    ok = err = 0
    for tid in table_ids:
        print(f"--- {tid} ---")
        success = reset_table_to_defaults(tid)
        if success:
            ok += 1
        else:
            err += 1
        print()

    print(f"📊 Ferdig: {ok} OK, {err} feil")


print("✅ Funksjoner definert")
print(f"   Kjører migrering av priority-kolonne hvis nødvendig...")
_ensure_config_columns()

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 5, Finished, Available, Finished, False)

✅ Funksjoner definert
   Kjører migrering av priority-kolonne hvis nødvendig...


In [4]:
# ============================================================================
# 1  SE ALLE TABELLER I CONFIG
# ============================================================================
# Ren visning – endrer ingenting. Viser alle tabellene som i dag følges av
# pipelinen, og når hver av dem sist ble lastet.

print("=" * 80)
print("1  SE ALLE TABELLER I CONFIG")
print("=" * 80)

if not spark.catalog.tableExists(CONFIG_TABLE):
    print(f"\nTabellen '{CONFIG_TABLE}' finnes ikke enda.")
    print("Kjor steg 9 (Batch insert) for aa opprette den forste gang.\n")
else:
    config_df = spark.sql(f"""
        SELECT
            table_id,
            table_name,
            frequency,
            category,
            lookback_periods,
            COALESCE(priority, 'NORMAL') AS priority,
            last_loaded_timestamp,
            CASE
                WHEN last_loaded_timestamp IS NULL THEN 'Aldri lastet'
                ELSE last_loaded_timestamp
            END AS status
        FROM {CONFIG_TABLE}
        ORDER BY priority DESC, category, table_name
    """)
    print(f"\nTotalt {config_df.count()} tabeller i config:\n")
    display(config_df)


StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 6, Finished, Available, Finished, False)

1  SE ALLE TABELLER I CONFIG

Totalt 53 tabeller i config:



SynapseWidget(Synapse.DataFrame, abcb01d1-a586-42bb-bf6a-91699c89ecc3)

In [5]:
# ============================================================================
# 2️⃣  LEGG TIL NY TABELL (enkelt)
# ============================================================================
# Registrerer én ny SSB-tabell i ssb_config, slik at pipelinen begynner å
# følge med på den. Henter automatisk navn, kategori og frekvens fra SSB
# sitt API – du trenger bare tabellnummeret. Legger samtidig inn tabellen i
# ssb_metadata-katalogen.

print("=" * 80)
print("2️⃣  LEGG TIL NY TABELL")
print("=" * 80)

if tabell_id is None:
    print("\n⏭️  Hoppet over – tabell_id er None")
    print("💡 Sett tabell_id = 'XXXXX' i parametercellen for å legge til en tabell")
else:
    add_new_table(
        tabell_id,
        override_category=kategori,
        override_lookback=lookback,
        override_priority=priority_ny,
    )

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 7, Finished, Available, Finished, False)

2️⃣  LEGG TIL NY TABELL
⚠️  Tabell 14289 finnes allerede i config – hopper over


In [6]:
# ============================================================================
# 3️⃣  OPPDATER EKSISTERENDE TABELL
# ============================================================================
# Endrer kategori, lookback eller priority for en tabell som allerede
# følges. Verdier du ikke oppgir selv (lar stå som None) hentes automatisk
# på nytt fra SSB.

print("=" * 80)
print("3️⃣  OPPDATER EKSISTERENDE TABELL")
print("=" * 80)

if tabell_id_update is None:
    print("\n⏭️  Hoppet over – tabell_id_update er None")
else:
    print(f"\n🔄 Oppdaterer tabell {tabell_id_update}...")
    _ensure_config_columns()

    if not _table_exists_in_config(tabell_id_update):
        print(f"❌ Tabell {tabell_id_update} finnes IKKE i config!")
    else:
        # Hent default-verdier fra API for felter som er None
        final_kategori = ny_kategori
        final_lookback = ny_lookback
        meta_row = None

        if ny_kategori is None or ny_lookback is None:
            print(f"📡 Henter default-verdier fra SSB API...")
            try:
                meta = fetch_table_metadata(tabell_id_update)
                meta_row = parse_cfg_row(tabell_id_update, meta)
                frequency = meta_row["update_frequency"] or meta_row["update_frequency_inferred"]
                if ny_kategori is None:
                    final_kategori = meta_row["subject_area"]
                    print(f"   ℹ️  category → {final_kategori} (fra API)")
                if ny_lookback is None:
                    final_lookback = get_default_lookback(frequency)
                    print(f"   ℹ️  lookback_periods → {final_lookback} (default for {frequency})")
            except Exception as e:
                print(f"❌ Feil ved henting fra SSB API: {e}")
                print("   Avbryter oppdatering – sett ny_kategori og ny_lookback manuelt")
                final_kategori = None

        if final_kategori is not None:
            set_dict = {
                "category":         F.lit(final_kategori),
                "lookback_periods": F.lit(int(final_lookback)),
            }
            if ny_priority is not None:
                valid_prio = ny_priority.upper() if ny_priority.upper() in VALID_PRIORITIES else "NORMAL"
                set_dict["priority"] = F.lit(valid_prio)
                print(f"   ℹ️  priority → {valid_prio}")

            DeltaTable.forName(spark, CONFIG_TABLE).update(
                condition=F.col("table_id") == tabell_id_update,
                set=set_dict,
            )

            print(f"\n✅ Tabell {tabell_id_update} oppdatert!")
            display(spark.table(CONFIG_TABLE).filter(F.col("table_id") == tabell_id_update))

            if meta_row is not None:
                upsert_metadata_row(meta_row)
                print(f"✅ ssb_metadata oppdatert for {tabell_id_update}")

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 8, Finished, Available, Finished, False)

3️⃣  OPPDATER EKSISTERENDE TABELL

⏭️  Hoppet over – tabell_id_update er None


In [7]:
# ============================================================================
# 4️⃣  SLETT EN TABELL
# ============================================================================
# Fjerner en tabell fra ssb_config for godt – pipelinen slutter å følge med
# på den. Rører ikke dataene som allerede er lastet, kun styringsraden.

print("=" * 80)
print("4️⃣  SLETT EN TABELL")
print("=" * 80)

if tabell_id_delete is None:
    print("\n⏭️  Hoppet over – tabell_id_delete er None")
else:
    print(f"\n🗑️  Sletter tabell {tabell_id_delete}...")

    if _table_exists_in_config(tabell_id_delete):
        print("📋 Rad som slettes:")
        display(spark.table(CONFIG_TABLE).filter(F.col("table_id") == tabell_id_delete))

    delete_table(tabell_id_delete)

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 9, Finished, Available, Finished, False)

4️⃣  SLETT EN TABELL

⏭️  Hoppet over – tabell_id_delete er None


In [8]:
# ============================================================================
# 5  VIS LOADING-HISTORIKK
# ============================================================================
# Ren visning – endrer ingenting. Sorterer tabellene etter når de sist ble
# lastet, med "aldri lastet" øverst, så du raskt ser om noe henger etter.

print("=" * 80)
print("5  LOADING-HISTORIKK")
print("=" * 80)

if not spark.catalog.tableExists(CONFIG_TABLE):
    print(f"\nTabellen '{CONFIG_TABLE}' finnes ikke enda.")
else:
    historikk_df = spark.sql(f"""
        SELECT
            table_id,
            table_name,
            frequency,
            COALESCE(priority, 'NORMAL') AS priority,
            last_loaded_timestamp,
            CASE
                WHEN last_loaded_timestamp IS NULL THEN 'ALDRI LASTET'
                ELSE last_loaded_timestamp
            END AS loading_status
        FROM {CONFIG_TABLE}
        ORDER BY
            CASE WHEN last_loaded_timestamp IS NULL THEN 0 ELSE 1 END,
            last_loaded_timestamp DESC
    """)
    print(f"\nLoading-status for alle tabeller:\n")
    display(historikk_df)


StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 10, Finished, Available, Finished, False)

5  LOADING-HISTORIKK

Loading-status for alle tabeller:



SynapseWidget(Synapse.DataFrame, d78a3412-5f5f-46dd-a406-ab150d6d4ef7)

In [9]:
# ============================================================================
# 6️⃣  UTILITY: Marker tabell som lastet
# ============================================================================
# Kjører ikke noe i seg selv – bare en påminnelse om at funksjonen
# mark_table_as_loaded() finnes og kan kalles fra andre steder om nødvendig.
# I den vanlige pipelinen skjer denne oppdateringen automatisk fra
# 05_ssb_standardize etter en vellykket kjøring.

print("=" * 80)
print("6️⃣  UTILITY: Marker tabell som lastet")
print("=" * 80)

print("""
💡 Kall denne funksjonen fra ingest-script etter vellykket last:

    mark_table_as_loaded(table_id)

Eksempel:
    # Etter at data er skrevet til Lakehouse:
    mark_table_as_loaded("07459")

# Kommenter inn for å teste:
# mark_table_as_loaded("12345")
""")

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 11, Finished, Available, Finished, False)

6️⃣  UTILITY: Marker tabell som lastet

💡 Kall denne funksjonen fra ingest-script etter vellykket last:

    mark_table_as_loaded(table_id)

Eksempel:
    # Etter at data er skrevet til Lakehouse:
    mark_table_as_loaded("07459")

# Kommenter inn for å teste:
# mark_table_as_loaded("12345")



In [10]:
# ============================================================================
# 7️⃣  RESET TIMESTAMP (force re-load neste kjøring)
# ============================================================================
# Nullstiller "sist lastet"-tidspunktet for én tabell. Brukes når du vil
# tvinge frem en full ny innlasting av tabellen selv om SSB ikke har
# publisert noe nytt siden sist – f.eks. etter en feilrettet bug.

print("=" * 80)
print("7️⃣  RESET TIMESTAMP")
print("=" * 80)

if tabell_id_reset is None:
    print("\n⏭️  Hoppet over – tabell_id_reset er None")
else:
    print(f"\n🔄 Resetter last_loaded_timestamp for {tabell_id_reset}...")

    if not _table_exists_in_config(tabell_id_reset):
        print(f"❌ Tabell {tabell_id_reset} finnes IKKE i config!")
    else:
        DeltaTable.forName(spark, CONFIG_TABLE).update(
            condition=F.col("table_id") == tabell_id_reset,
            set={"last_loaded_timestamp": F.lit(None).cast(StringType())},
        )
        print(f"✅ {tabell_id_reset}: last_loaded_timestamp → NULL")
        print(f"   Tabellen vil bli lastet som 'first_load' neste gang Update Detector kjører")

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 12, Finished, Available, Finished, False)

7️⃣  RESET TIMESTAMP

⏭️  Hoppet over – tabell_id_reset er None


In [11]:
# ============================================================================
# 8️⃣  RESET TIL DEFAULT (batch fra liste)
# ============================================================================
# Samme som steg 3 (oppdater), men for en liste med flere tabeller samtidig
# – henter fersk kategori og lookback fra SSB for hver av dem.

print("=" * 80)
print("8️⃣  RESET TIL DEFAULT")
print("=" * 80)

if not table_list_default:
    print("\n⏭️  Hoppet over – table_list_default er tom")
    print("💡 Sett table_list_default = ['12345', '67890'] for å resette")
else:
    print(f"\n📋 Resetter {len(table_list_default)} tabeller til default...\n")
    for tid in table_list_default:
        print(f"--- {tid} ---")
        reset_table_to_defaults(tid)
        print()
    print("✅ Ferdig med reset!")

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 13, Finished, Available, Finished, False)

8️⃣  RESET TIL DEFAULT

⏭️  Hoppet over – table_list_default er tom
💡 Sett table_list_default = ['12345', '67890'] for å resette


In [12]:
# ============================================================================
# 9️⃣  BATCH INSERT (legg til flere tabeller)
# ============================================================================
# Samme som steg 2 (legg til), men for flere tabellnumre på én gang – nyttig
# ved førstegangs oppsett av pipelinen med mange tabeller.

print("=" * 80)
print("9️⃣  BATCH INSERT")
print("=" * 80)

if not table_list:
    print("\n⏭️  Hoppet over – table_list er tom")
    print("💡 Sett table_list = ['12345', '67890'] for å legge til flere tabeller")
else:
    print(f"\n📋 Legger til {len(table_list)} tabeller...\n")
    ok = skipped = 0
    for tid in table_list:
        print(f"--- {tid} ---")
        result = add_new_table(
            tid,
            override_category=kategori,
            override_lookback=lookback,
            override_priority=priority_ny,
        )
        if result:
            ok += 1
        else:
            skipped += 1
        print()

    print(f"📊 Resultat: {ok} lagt til, {skipped} hoppet over")

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 14, Finished, Available, Finished, False)

9️⃣  BATCH INSERT

⏭️  Hoppet over – table_list er tom
💡 Sett table_list = ['12345', '67890'] for å legge til flere tabeller


In [ ]:
# ============================================================================
# 🔟  BATCH SLETT (flere tabeller)
# ============================================================================
# Samme som steg 4 (slett), men for flere tabellnumre på én gang – nyttig
# hvis flere tabeller skal fjernes fra pipelinen samtidig.

print("=" * 80)
print("🔟  BATCH SLETT")
print("=" * 80)

if not table_list_delete:
    print("\n⏭️  Hoppet over – table_list_delete er tom")
    print("💡 Sett table_list_delete = ['12345', '67890'] for å slette flere tabeller")
else:
    delete_tables(table_list_delete)


In [13]:
# ============================================================================
# 1️⃣1️⃣ RESET ALLE TABELLER TIL DEFAULT
# ============================================================================
# Det kraftigste steget – resetter kategori og lookback for absolutt alle
# tabeller i config. Derfor krever den et eksplisitt confirm=True i koden
# under, den kjører ikke ved et uhell selv om du trykker "Run All".

print("=" * 80)
print("1️⃣1️⃣ RESET ALLE TABELLER TIL DEFAULT")
print("=" * 80)

print("""
⚠️  ADVARSEL: Resetter kategori og lookback_periods for ALLE tabeller!
   last_loaded_timestamp og priority røres IKKE.

For å kjøre:
   reset_all_tables_to_defaults(confirm=True)
""")

# Kommenter inn for å kjøre:
# reset_all_tables_to_defaults(confirm=True)

StatementMeta(, 0e024090-e827-4873-bafe-b45b83e1a5ce, 15, Finished, Available, Finished, False)

🔟 RESET ALLE TABELLER TIL DEFAULT

⚠️  ADVARSEL: Resetter kategori og lookback_periods for ALLE tabeller!
   last_loaded_timestamp og priority røres IKKE.

For å kjøre:
   reset_all_tables_to_defaults(confirm=True)

